In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

print(sys.executable)  # should end in sec-10k-rag/.venv/bin/python

In [ ]:
from pathlib import Path

from sec_10k_rag.parsing.html_text import html_to_text
from sec_10k_rag.parsing.sections import split_into_items

In [ ]:
html: bytes = Path("../data/raw/TSLA/2025/10k.html").read_bytes()
text: str = html_to_text(html)
print(text[:2000])

In [ ]:
print(text[2000:6000])

In [ ]:
from sec_10k_rag.models import Heading, RawSection
from sec_10k_rag.parsing.sections import (
    body_start,
    find_headings,
    pick_body_headings,
)

In [ ]:
print(f"{len(text):,} characters")

In [ ]:
headings: list[Heading] = find_headings(text)
print(len(headings), "heading-like lines")

In [ ]:
print(headings[:10])

In [ ]:
for h in headings:
    line: str = text[h.start : text.find("\n", h.start)]
    print(f"{h.start:>8,}  {h.item:<3} {line}")

In [ ]:
start: int = body_start(headings)
print("body starts at heading #", start, "->", headings[start])

In [ ]:
chosen: list[Heading] = pick_body_headings(headings[start:])
print([h.item for h in chosen])

In [ ]:
sections: list[RawSection] = split_into_items(text)
for s in sections:
    print(f"{s.item:<3} {len(s.text):>9,}  {s.title:<45} {s.text[:50]!r}")

In [ ]:
from sec_10k_rag.parsing.sgml import extract_10k_html, split_documents

# the start of the 10-K's HTML

In [ ]:
sub: str = Path("../data/raw/TSLA/2025/full_submission.txt").read_text(
    encoding="utf-8", errors="replace"
)
docs = split_documents(sub)

In [ ]:
print(len(docs), "documents")
print([d.doc_type for d in docs])  # 10-K, EX-..., GRAPHIC, XML, ...
print(extract_10k_html(sub)[:500])

In [ ]:
def test_missing_item_is_skipped(fixtures_dir: Path) -> None:
    file_path = fixtures_dir / "missing_items_in_filing.txt"
    test_data = file_path.read_text(encoding="utf-8")
    sections = split_into_items(test_data)
    expected_items = ["1", "7"]
    assert [section.item for section in sections] == expected_items
    assert sections[0].text.startswith("Item 1. Business\nWe design")
    assert sections[-1].text.endswith("Total revenues grew 3% in 2025.")

In [ ]:
# test_missing_item_is_skipped(Path(__file__).parent / "..")
# from pathlib import Path
# print(Path.cwd().parent / "fixtures/missing_items_in_filing.txt")
test_missing_item_is_skipped(Path.cwd().parent / "tests/fixtures")